# Approximated State Preparation in native FlagQuantum

Complete local Notebook workflow: reproduce seeded 256-element data, all ten bounds, executed resource/error measurements and saved plot. Retained six-bit Qmod counterpart remains separately available through build_program.

Current exact native preparation satisfies all requested bounds and yields a flat resource curve; no approximation compression claim. Record native instruction-basis depth and CX counts, not Classiq custom CX/U transpilation metrics. NumPy RandomState(1) preserves the source data sequence without changing global random state. No cloud, remote timeout or hardware claim.

Define the native quantum functions below, synthesize them locally, inspect the executed results, and compare the complete complex state with an independent Circuit.

See [the native Python entry](bound_sweep.py).

## Native imports for bound_sweep

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import torch
import flagquantum as fq
from flagquantum import synth
from flagquantum.compiler import estimate_resources
from flagquantum.synth import Output, QArray, QBit, prepare_amplitudes, qfunc

## PreparationPoint

In [ ]:
@dataclass(frozen=True)
class PreparationPoint:
    """Executed preparation and measured errors at one requested upper bound."""

    bound: float
    program: synth.Program
    amplitude_l2_error: float
    probability_l2_error: float
    depth: int
    cx_count: int

## prepare

In [ ]:
@qfunc
def prepare(amplitudes: list[float], bound: float, out: Output[QArray[QBit]]) -> None:
    prepare_amplitudes(amplitudes, bound, out)

## source amplitudes

In [ ]:
def source_amplitudes() -> list[float]:
    """Reproduce the source seed, cubic trend, perturbation and normalization."""
    x = np.linspace(-1, 1, 256)
    trend = 0.6 - 0.4 * x + 0.8 * x**2 - 0.3 * x**3
    amplitudes = trend + 0.05 * np.random.RandomState(1).randn(256)
    amplitudes -= np.mean(amplitudes)
    return [float(value) for value in amplitudes / np.linalg.norm(amplitudes)]

## run sweep

In [ ]:
def run_sweep() -> tuple[PreparationPoint, ...]:
    """Execute all ten original bounds and measure actual compiler resources.

    The current exact kernel satisfies every bound; it does not trade accuracy
    for fewer gates. Resource rows use the native instruction basis, not the
    source platform's custom CX/U transpilation basis.
    """
    amplitudes = source_amplitudes()
    expected = torch.tensor(amplitudes, dtype=torch.complex128)
    points = []
    for value in np.linspace(0.0, 0.3, 10):
        bound = float(value)
        program = synth.synthesize(
            prepare,
            classical_inputs={"amplitudes": amplitudes, "bound": bound},
            max_qubits=8,
            max_workspace=0,
            memory_limit_bytes=64 * 1024**2,
        )
        state = fq.run(program).statevector().flatten()
        amplitude_error = float(torch.linalg.vector_norm(state - expected))
        probability_error = float(
            torch.linalg.vector_norm(state.abs().square() - expected.abs().square())
        )
        resources = estimate_resources(program.circuit)
        points.append(
            PreparationPoint(
                bound,
                program,
                amplitude_error,
                probability_error,
                int(resources.depth),
                int(resources.operation_counts.get("cx", 0)),
            )
        )
    return tuple(points)

## plot sweep

In [ ]:
def plot_sweep(points: tuple[PreparationPoint, ...], output: Path) -> None:
    """Save measured native depth and CX count without claiming compression."""
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots()
    bounds = [point.bound for point in points]
    axes.plot(bounds, [point.depth for point in points], "o-", label="Native depth")
    axes.plot(
        bounds, [point.cx_count for point in points], "*-", label="Native CX count"
    )
    axes.set_xlabel("Requested amplitude L2 error upper bound")
    axes.set_ylabel("Native depth, CX count")
    axes.set_title("Exact preparation: no approximation savings")
    axes.legend()
    figure.tight_layout()
    figure.savefig(output, dpi=150)
    plt.close(figure)

## run example

In [ ]:
def run_example(output: Path = Path("native_amplitude_bound_sweep.png")) -> None:
    """Run data generation, all bound experiments, resource reporting and plot."""
    points = run_sweep()
    for point in points:
        print(
            f"bound={point.bound:.6f} depth={point.depth} CX={point.cx_count} "
            f"amplitude_L2={point.amplitude_l2_error:.3g} "
            f"probability_L2={point.probability_l2_error:.3g}"
        )
    plot_sweep(points, output)
    print(f"Saved native resource plot to {output}")

## Execute the complete native workflow

Run every cell in order. Any sampling is local and seeded; resource reports come from the actual native program. Read the reported approximation limits and optimization outcomes alongside the numerical results.

In [ ]:
output = Path("native_amplitude_bound_sweep.png")

In [ ]:
run_example(output)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(output)))

## Verify absolute amplitudes and clean workspace

The independent Circuit uses the same declared register order. The comparison checks every complex amplitude without global-phase alignment and verifies zero probability in hidden work qubits. This is a local numerical check.


In [ ]:
import torch
from flagquantum import synth
from examples.synthesis.compare_states import compare_states


def householder_reference(amplitudes: list[float]) -> fq.Circuit:
    target = torch.tensor(amplitudes, dtype=torch.complex128)
    target = target / torch.linalg.vector_norm(target)
    difference = -target.clone()
    difference[0] += 1
    reflection = torch.eye(target.numel(), dtype=torch.complex128) - 2 * torch.outer(
        difference, difference.conj()
    ) / torch.vdot(difference, difference)
    width = target.numel().bit_length() - 1
    return fq.Circuit(width, dtype=torch.complex128).any(
        *range(width), matrix=reflection
    )


reference = householder_reference(source_amplitudes())
for bound in np.linspace(0.0, 0.3, 10):
    program = synth.synthesize(
        prepare,
        classical_inputs={"amplitudes": source_amplitudes(), "bound": float(bound)},
        max_qubits=8,
        max_workspace=0,
        memory_limit_bytes=64 * 1024**2,
    )
    compare_states(program, reference, ("out",))
print("Independent full-state comparison: passed")
